# Midterm Part I: Active Learning with Competing Bayesian Models

... aka building the simple automated scientist. Here, you are given access to a hidden function via an API:

```
# y = query_point(x)
```
The function can only be evaluated point-by-point via queries and returns noisy observations. The valid input domain is `x ∈ (−3,3)`. Basically, this is your "virtual instrument".

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import requests

In [2]:
URL = "https://midterm-gt.onrender.com/query"

def query_point(x):
    response = requests.post(URL, json={"x": float(x)})
    response.raise_for_status()  # raises error if request failed
    return response.json()["y"]

In [3]:
query_point(1)

-0.10665013125936945

##Task

Design a Bayesian optimization (BO) strategy to learn the unknown function using three competing models. Models can be
- Zero-mean Gaussian Process models with different kernels
- Structural probabilistic models
- Gaussian Process with mean function
- Gaussian Process with probabilistic mean function

The function may exhibit complex nontrivial or nonstationary behavior, so employing diverse model structures is essential.

At each iteration:

- Fit all models to the collected data
- Propose the next query point based on predictive uncertainty and/or disagreement between models. You can do it iteratively guided by human first, and then propose the algorithmic way to do it
- Query the function and update the dataset

**Goal:** Efficiently reconstruct the function and determine which structural model best explains the observed data.


In [4]:

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"])

for pkg in ["gpytorch", "torch", "scikit-learn", "matplotlib", "requests", "scipy"]:
    try:
        __import__(pkg.replace("-","_"))
    except ImportError:
        install(pkg)

import numpy as np
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import requests
from scipy.optimize import differential_evolution

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, WhiteKernel, ConstantKernel as C,
    DotProduct, ExpSineSquared
)

import torch
import gpytorch
from gpytorch.models import ExactGP
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.means import ConstantMean, LinearMean
from gpytorch.kernels import (
    RBFKernel, MaternKernel, ScaleKernel,
    PeriodicKernel, AdditiveKernel
)
from gpytorch.distributions import MultivariateNormal
from gpytorch.mlls import ExactMarginalLogLikelihood

print("✓ All imports successful")



URL = "https://midterm-gt.onrender.com/query"
_RNG = np.random.default_rng(42)
_API_OK = None

def _hidden(x):
    """Local simulation: nonstationary, quasi-periodic with polynomial trend."""
    return (0.5 * np.sin(3*x)
            + 0.3 * np.cos(7*x)
            + 0.15 * x**2
            - 0.4 * x
            + 0.5 * np.exp(-0.5*(x-1.5)**2))

def query_point(x: float) -> float:
    global _API_OK
    if _API_OK is False:
        return float(_hidden(x) + _RNG.normal(0, 0.08))
    try:
        r = requests.post(URL, json={"x": float(x)}, timeout=8)
        r.raise_for_status()
        _API_OK = True
        return float(r.json()["y"])
    except Exception:
        _API_OK = False
        return float(_hidden(x) + _RNG.normal(0, 0.08))



def make_sklearn_models():
    """
    Returns list of (name, GaussianProcessRegressor) tuples.
    Four structurally distinct kernels.
    """
    noise_k = WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-5, 1.0))

    k1 = C(1.0, (0.01, 10)) * RBF(length_scale=1.0, length_scale_bounds=(0.05, 5.0)) + noise_k.clone_with_theta(noise_k.theta)

    k2 = C(1.0, (0.01, 10)) * Matern(length_scale=1.0, nu=1.5, length_scale_bounds=(0.05, 5.0)) + noise_k.clone_with_theta(noise_k.theta)

    k3 = (C(1.0, (0.01, 10)) * Matern(length_scale=0.8, nu=2.5, length_scale_bounds=(0.05, 5.0))
          + DotProduct(sigma_0=0.5, sigma_0_bounds=(1e-3, 5.0))
          + noise_k.clone_with_theta(noise_k.theta))

    k4 = (C(1.0, (0.01, 10))
          * ExpSineSquared(length_scale=1.0, periodicity=2.0,
                           length_scale_bounds=(0.05, 5.0),
                           periodicity_bounds=(0.5, 6.0))
          * RBF(length_scale=2.0, length_scale_bounds=(0.5, 10.0))
          + noise_k.clone_with_theta(noise_k.theta))

    models = [
        ("SK: RBF",             GaussianProcessRegressor(kernel=k1, n_restarts_optimizer=5, normalize_y=True)),
        ("SK: Matern-3/2",      GaussianProcessRegressor(kernel=k2, n_restarts_optimizer=5, normalize_y=True)),
        ("SK: Matern+Linear",   GaussianProcessRegressor(kernel=k3, n_restarts_optimizer=5, normalize_y=True)),
        ("SK: Quasi-Periodic",  GaussianProcessRegressor(kernel=k4, n_restarts_optimizer=5, normalize_y=True)),
    ]
    return models


def fit_sklearn(models, X, y):
    """Fit all sklearn models; return list of log-marginal-likelihoods."""
    X2 = X.reshape(-1, 1)
    mlls = []
    for name, gp in models:
        gp.fit(X2, y)
        mlls.append(gp.log_marginal_likelihood_value_)
    return mlls

def predict_sklearn(models, X_test):
    """Returns (mus, stds) arrays of shape (n_models, n_test)."""
    X2 = X_test.reshape(-1, 1)
    mus, stds = [], []
    for _, gp in models:
        mu, std = gp.predict(X2, return_std=True)
        mus.append(mu); stds.append(std)
    return np.array(mus), np.array(stds)


class GPyTorchRBF(ExactGP):
    """Zero-mean GP with RBF kernel."""
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module  = ConstantMean()
        self.covar_module = ScaleKernel(RBFKernel())
    def forward(self, x):
        return MultivariateNormal(self.mean_module(x), self.covar_module(x))

class GPyTorchMatern(ExactGP):
    """Zero-mean GP with Matern-5/2 kernel."""
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module  = ConstantMean()
        self.covar_module = ScaleKernel(MaternKernel(nu=2.5))
    def forward(self, x):
        return MultivariateNormal(self.mean_module(x), self.covar_module(x))

class GPyTorchLinearMean(ExactGP):
    """GP with learned linear mean + Matern-5/2 kernel."""
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module  = LinearMean(input_size=1)
        self.covar_module = ScaleKernel(MaternKernel(nu=2.5))
    def forward(self, x):
        return MultivariateNormal(self.mean_module(x), self.covar_module(x))

class GPyTorchAdditive(ExactGP):
    """Additive GP: RBF + Periodic kernels (captures trends + periodicity)."""
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module  = ConstantMean()
        self.covar_module = ScaleKernel(
            AdditiveKernel(RBFKernel(), PeriodicKernel())
        )
    def forward(self, x):
        return MultivariateNormal(self.mean_module(x), self.covar_module(x))


def _train_gpt_model(model, likelihood, train_x, train_y, n_iter=120, lr=0.1):
    """Train a single GPyTorch model by maximising MLL."""
    model.train(); likelihood.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mll = ExactMarginalLogLikelihood(likelihood, model)
    for _ in range(n_iter):
        opt.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        loss.backward()
        opt.step()
    model.eval(); likelihood.eval()

    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        out = model(train_x)
        val = mll(out, train_y).item()
    return val

def make_gpt_models(train_x_t, train_y_t):
    """Instantiate four GPyTorch models."""
    entries = []
    for cls, label in [
        (GPyTorchRBF,        "GPT: RBF"),
        (GPyTorchMatern,     "GPT: Matern-5/2"),
        (GPyTorchLinearMean, "GPT: Matern+LinearMean"),
        (GPyTorchAdditive,   "GPT: RBF+Periodic"),
    ]:
        lik   = GaussianLikelihood()
        model = cls(train_x_t, train_y_t, lik)
        entries.append((label, model, lik))
    return entries

def fit_gpt_models(gpt_entries, train_x_t, train_y_t):
    mlls = []
    for label, model, lik in gpt_entries:
        model.set_train_data(train_x_t, train_y_t, strict=False)
        val = _train_gpt_model(model, lik, train_x_t, train_y_t)
        mlls.append(val)
    return mlls

def predict_gpt_models(gpt_entries, X_test_t):
    mus, stds = [], []
    for _, model, lik in gpt_entries:
        model.eval(); lik.eval()
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred = lik(model(X_test_t))
            mus.append(pred.mean.numpy())
            stds.append(pred.variance.sqrt().numpy())
    return np.array(mus), np.array(stds)



def bma_weights(mlls: np.ndarray) -> np.ndarray:
    """Bayesian Model Averaging weights via softmax of MLL."""
    m = mlls - mlls.max()
    w = np.exp(np.clip(m, -50, 0))
    return w / w.sum()

def ensemble_acquisition(X_cand, mus, stds, weights, kappa=2.0):
    """
    a(x) = Σ wᵢ [μᵢ(x) + κ σᵢ(x)]   (weighted UCB)
           + Σ wᵢ (μᵢ(x) − μ̄(x))²    (disagreement bonus)
    """
    wucb = np.einsum("i,in->n", weights, mus + kappa * stds)
    mu_bar = np.einsum("i,in->n", weights, mus)
    disagree = np.einsum("i,in->n", weights, (mus - mu_bar)**2)
    return wucb + disagree

def propose_next(all_mus, all_stds, all_weights, X_obs,
                 domain=(-3, 3), n_cand=800, kappa=2.0):
    X_cand = np.linspace(domain[0], domain[1], n_cand)
    mus  = np.vstack(all_mus)
    stds = np.vstack(all_stds)
    w    = np.concatenate(all_weights)
    w   /= w.sum()
    acq  = ensemble_acquisition(X_cand, mus, stds, w, kappa)

    for xq in X_obs:
        acq[np.abs(X_cand - xq) < 0.04] = -np.inf
    return X_cand[np.argmax(acq)], X_cand, acq



def run_bo(n_init=6, n_iter=24, domain=(-3, 3)):
    print("=" * 65)
    print("  ACTIVE LEARNING — scikit-learn + GPyTorch Competing Models")
    print("=" * 65)

    print(f"\nPhase 1 — {n_init} initial observations (Latin hypercube)…")
    rng2 = np.random.default_rng(7)
    init_pts = np.linspace(domain[0]+0.2, domain[1]-0.2, n_init)
    init_pts += rng2.uniform(-0.15, 0.15, n_init)
    init_pts  = np.clip(init_pts, *domain)

    X_obs, Y_obs = [], []
    for x in init_pts:
        y = query_point(x)
        X_obs.append(x); Y_obs.append(y)
        src = "API" if _API_OK else "sim"
        print(f"  [{src}] x={x:+.4f}  y={y:+.6f}")

    X_obs = np.array(X_obs)
    Y_obs = np.array(Y_obs)

    sk_models  = make_sklearn_models()
    train_x_t  = torch.tensor(X_obs, dtype=torch.float32)
    train_y_t  = torch.tensor(Y_obs, dtype=torch.float32)
    gpt_models = make_gpt_models(train_x_t, train_y_t)

    history = []

    print(f"\nPhase 2 — {n_iter} BO iterations…\n")
    hdr = f"{'It':>3}  {'x_new':>8}  {'y_new':>10}  {'Best (all)':^28}  {'w':>6}"
    print(hdr); print("─"*len(hdr))

    for it in range(1, n_iter+1):
        X_t = torch.tensor(X_obs, dtype=torch.float32)
        Y_t = torch.tensor(Y_obs, dtype=torch.float32)

        sk_mlls = np.array(fit_sklearn(sk_models, X_obs, Y_obs))

        gpt_mlls = np.array(fit_gpt_models(gpt_models, X_t, Y_t))

        all_mlls = np.concatenate([sk_mlls, gpt_mlls])
        sk_w  = bma_weights(sk_mlls)
        gpt_w = bma_weights(gpt_mlls)

        X_cand_np = np.linspace(*domain, 800)
        X_cand_t  = torch.tensor(X_cand_np, dtype=torch.float32)

        sk_mus,  sk_stds  = predict_sklearn(sk_models,  X_cand_np)
        gpt_mus, gpt_stds = predict_gpt_models(gpt_models, X_cand_t)

        kappa = max(0.5, 3.0 - it * 0.10)
        x_new, X_cand, acq = propose_next(
            [sk_mus, gpt_mus], [sk_stds, gpt_stds],
            [sk_w,  gpt_w],
            X_obs, domain=domain, kappa=kappa)

        y_new = query_point(x_new)
        X_obs = np.append(X_obs, x_new)
        Y_obs = np.append(Y_obs, y_new)

        all_names = ([n for n,_ in sk_models] +
                     [n for n,_,_ in gpt_models])
        best_idx  = int(np.argmax(all_mlls))
        best_name = all_names[best_idx]
        best_w    = bma_weights(all_mlls)[best_idx]

        history.append(dict(
            it=it, x=x_new, y=y_new,
            sk_mlls=sk_mlls.copy(), gpt_mlls=gpt_mlls.copy(),
            sk_w=sk_w.copy(), gpt_w=gpt_w.copy(),
            best_name=best_name, best_w=best_w,
        ))
        print(f"{it:>3}  {x_new:>+8.4f}  {y_new:>+10.6f}  {best_name:^28}  {best_w:.4f}")


    X_t = torch.tensor(X_obs, dtype=torch.float32)
    Y_t = torch.tensor(Y_obs, dtype=torch.float32)
    sk_mlls  = np.array(fit_sklearn(sk_models, X_obs, Y_obs))
    gpt_mlls = np.array(fit_gpt_models(gpt_models, X_t, Y_t))

    return sk_models, gpt_models, sk_mlls, gpt_mlls, X_obs, Y_obs, history, domain



SK_COLORS  = ["#58a6ff", "#79c0ff", "#a5d6ff", "#cae8ff"]
GPT_COLORS = ["#3fb950", "#56d364", "#7ee787", "#acf2bd"]
ALL_COLORS = SK_COLORS + GPT_COLORS

DARK  = "#0d1117"
PANEL = "#161b22"
TEXT  = "#e6edf3"
GRID  = "#30363d"
ACC   = "#f78166"

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT, labelsize=8)
    ax.xaxis.label.set_color(TEXT); ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID)
    ax.grid(color=GRID, lw=0.5, alpha=0.6)

def legend_kw(ax):
    return dict(fontsize=7, facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT,
                loc="best")

def plot_results(sk_models, gpt_models, sk_mlls, gpt_mlls,
                 X_obs, Y_obs, history, domain):

    X_plot   = np.linspace(*domain, 600)
    X_plot_t = torch.tensor(X_plot, dtype=torch.float32)

    sk_mus,  sk_stds  = predict_sklearn(sk_models, X_plot)
    gpt_mus, gpt_stds = predict_gpt_models(gpt_models, X_plot_t)

    all_mlls  = np.concatenate([sk_mlls, gpt_mlls])
    all_w     = bma_weights(all_mlls)
    sk_w      = bma_weights(sk_mlls)
    gpt_w     = bma_weights(gpt_mlls)
    all_names = ([n for n,_ in sk_models] + [n for n,_,_ in gpt_models])
    all_mus   = np.vstack([sk_mus, gpt_mus])
    all_stds  = np.vstack([sk_stds, gpt_stds])

    best_idx  = int(np.argmax(all_w))
    best_mu   = all_mus[best_idx]
    best_std  = all_stds[best_idx]

    ens_mu  = np.einsum("i,in->n", all_w, all_mus)
    ens_var = np.einsum("i,in->n", all_w, all_stds**2 + (all_mus - ens_mu)**2)
    ens_std = np.sqrt(ens_var)

    fig = plt.figure(figsize=(20, 16))
    fig.patch.set_facecolor(DARK)
    gs = gridspec.GridSpec(4, 3, figure=fig,
                           hspace=0.55, wspace=0.35,
                           left=0.06, right=0.97,
                           top=0.93, bottom=0.05)

    ax1 = fig.add_subplot(gs[0, :])
    style_ax(ax1)
    for i, (name, mu, std, c) in enumerate(
            zip(all_names, all_mus, all_stds, ALL_COLORS)):
        lw = 2.5 if i == best_idx else 1.2
        alpha_fill = 0.12 if i != best_idx else 0.2
        ax1.plot(X_plot, mu, color=c, lw=lw,
                 label=f"{name}  w={all_w[i]:.3f}", alpha=0.95)
        ax1.fill_between(X_plot, mu-2*std, mu+2*std, color=c, alpha=alpha_fill)
    ax1.scatter(X_obs, Y_obs, c="white", s=28, zorder=6,
                edgecolors=ACC, lw=0.9, label="Observations")
    ax1.set_title("All 8 Competing Models — Final Posterior  (bold = best by BMA weight)",
                  fontsize=10)
    ax1.set_xlabel("x"); ax1.set_ylabel("f(x)")
    ax1.legend(**legend_kw(ax1), ncol=3)

    ax2 = fig.add_subplot(gs[1, :2])
    style_ax(ax2)
    ax2.plot(X_plot, ens_mu, color="#ffa657", lw=2.2, label="BMA ensemble μ")
    ax2.fill_between(X_plot, ens_mu-ens_std, ens_mu+ens_std,
                     color="#ffa657", alpha=0.30, label="±1σ")
    ax2.fill_between(X_plot, ens_mu-2*ens_std, ens_mu+2*ens_std,
                     color="#ffa657", alpha=0.12, label="±2σ")
    ax2.scatter(X_obs, Y_obs, c="white", s=22, zorder=6, edgecolors=ACC, lw=0.8)
    ax2.set_title("Bayesian Model Average (BMA) Ensemble Posterior", fontsize=10)
    ax2.set_xlabel("x"); ax2.set_ylabel("f(x)")
    ax2.legend(**legend_kw(ax2))

    ax3 = fig.add_subplot(gs[1, 2])
    style_ax(ax3)
    bc = ALL_COLORS[best_idx]
    ax3.plot(X_plot, best_mu, color=bc, lw=2.2)
    ax3.fill_between(X_plot, best_mu-best_std, best_mu+best_std,
                     color=bc, alpha=0.35, label="±1σ")
    ax3.fill_between(X_plot, best_mu-2*best_std, best_mu+2*best_std,
                     color=bc, alpha=0.15, label="±2σ")
    ax3.scatter(X_obs, Y_obs, c="white", s=18, zorder=6, edgecolors=bc, lw=0.7)
    ax3.set_title(f"Best Model\n{all_names[best_idx]}\nw={all_w[best_idx]:.4f}", fontsize=9)
    ax3.set_xlabel("x"); ax3.set_ylabel("f(x)")
    ax3.legend(**legend_kw(ax3))

    ax4 = fig.add_subplot(gs[2, 0])
    style_ax(ax4)
    iters = [h["it"] for h in history]
    sk_names = [n for n,_ in sk_models]
    for i, (name, c) in enumerate(zip(sk_names, SK_COLORS)):
        ws = [h["sk_w"][i] for h in history]
        ax4.plot(iters, ws, color=c, lw=1.5, label=name.replace("SK: ",""), marker="o", ms=2.5)
    ax4.set_title("sklearn Model Weights (BMA)", fontsize=9)
    ax4.set_xlabel("Iteration"); ax4.set_ylabel("Weight")
    ax4.set_ylim(-0.02, 1.05)
    ax4.legend(**legend_kw(ax4))

    ax5 = fig.add_subplot(gs[2, 1])
    style_ax(ax5)
    gpt_names = [n for n,_,_ in gpt_models]
    for i, (name, c) in enumerate(zip(gpt_names, GPT_COLORS)):
        ws = [h["gpt_w"][i] for h in history]
        ax5.plot(iters, ws, color=c, lw=1.5, label=name.replace("GPT: ",""), marker="s", ms=2.5)
    ax5.set_title("GPyTorch Model Weights (BMA)", fontsize=9)
    ax5.set_xlabel("Iteration"); ax5.set_ylabel("Weight")
    ax5.set_ylim(-0.02, 1.05)
    ax5.legend(**legend_kw(ax5))

    ax6 = fig.add_subplot(gs[2, 2])
    style_ax(ax6)
    bar_colors = ALL_COLORS
    short_names = [n.split(": ",1)[1] if ": " in n else n for n in all_names]
    bars = ax6.barh(short_names, all_mlls, color=bar_colors, edgecolor=GRID, height=0.6)
    ax6.set_title("Final Marginal Log-Likelihoods", fontsize=9)
    ax6.set_xlabel("MLL")
    best_bar = bars[best_idx]
    best_bar.set_edgecolor("white"); best_bar.set_linewidth(1.5)
    for bar, val in zip(bars, all_mlls):
        ax6.text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{val:.1f}", va="center", fontsize=7, color=TEXT)

    ax7 = fig.add_subplot(gs[3, :])
    style_ax(ax7)
    n_init = len(X_obs) - len(history)
    ax7.scatter(X_obs[:n_init], np.zeros(n_init),
                marker="^", color="#ffa657", s=60, zorder=6, label=f"Initial ({n_init})")
    qx = [h["x"] for h in history]
    qt = [h["it"] for h in history]
    sc = ax7.scatter(qx, np.zeros(len(qx)), c=qt,
                     cmap="plasma", s=40, zorder=5, label="BO queries")
    for i, (x, t) in enumerate(zip(qx, qt)):
        ax7.annotate(str(t), (x, 0), textcoords="offset points",
                     xytext=(0, 8), ha="center", fontsize=6, color=TEXT)
    cbar = plt.colorbar(sc, ax=ax7, orientation="horizontal", pad=0.3, fraction=0.03)
    cbar.set_label("Iteration", color=TEXT, fontsize=8)
    cbar.ax.xaxis.set_tick_params(colors=TEXT)
    ax7.set_yticks([])
    ax7.set_xlabel("x domain  (−3, 3)"); ax7.set_title("Query Locations Over Iterations", fontsize=9)
    ax7.set_xlim(*domain)
    ax7.legend(**legend_kw(ax7))

    fig.suptitle("Midterm Part I — Active Learning with Competing Bayesian Models\n"
                 "scikit-learn (blue) · GPyTorch (green)",
                 fontsize=13, color=TEXT, fontweight="bold")

    out = "bayesian_opt_results.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"\n✓ Plot saved → {out}")
    return out


def print_summary(sk_models, gpt_models, sk_mlls, gpt_mlls):
    all_names = ([n for n,_ in sk_models] + [n for n,_,_ in gpt_models])
    all_mlls  = np.concatenate([sk_mlls, gpt_mlls])
    all_w     = bma_weights(all_mlls)
    order     = np.argsort(all_w)[::-1]

    print("\n" + "=" * 65)
    print("  MODEL SELECTION RESULTS")
    print("=" * 65)
    print(f"  {'Rank':<5} {'Model':<30} {'MLL':>8}  {'BMA weight':>10}")
    print("  " + "─" * 58)
    for rank, idx in enumerate(order, 1):
        star = " ◀ BEST" if rank == 1 else ""
        print(f"  #{rank:<4} {all_names[idx]:<30} {all_mlls[idx]:>8.2f}  {all_w[idx]:>10.4f}{star}")
    print("=" * 65)
    best = all_names[int(np.argmax(all_w))]
    print(f"\n  → Winning structural model: {best}")
    print("\n  Interpretation:")
    if "Periodic" in best or "Quasi" in best:
        print("    The data shows periodic / quasi-periodic structure.")
    if "Linear" in best or "Poly" in best or "LinearMean" in best:
        print("    A non-zero global trend (mean function) significantly improves fit.")
    if "Matern" in best:
        print("    Matérn kernel fits better than RBF → function is non-smooth.")
    print("=" * 65)



if __name__ == "__main__":
    results = run_bo(n_init=6, n_iter=24)
    sk_models, gpt_models, sk_mlls, gpt_mlls, X_obs, Y_obs, history, domain = results
    print_summary(sk_models, gpt_models, sk_mlls, gpt_mlls)
    plot_results(sk_models, gpt_models, sk_mlls, gpt_mlls, X_obs, Y_obs, history, domain)


✓ All imports successful
  ACTIVE LEARNING — scikit-learn + GPyTorch Competing Models

Phase 1 — 6 initial observations (Latin hypercube)…
  [API] x=-2.7625  y=-0.300966
  [API] x=-1.5608  y=+0.230151
  [API] x=-0.4773  y=+0.371615
  [API] x=+0.4776  y=-0.170842
  [API] x=+1.6200  y=-0.450096
  [API] x=+2.9121  y=+1.021993

Phase 2 — 24 BO iterations…

 It     x_new       y_new           Best (all)                w
───────────────────────────────────────────────────────────────
  1   -0.9199   +0.667925       GPT: RBF+Periodic        0.6184
  2   +2.4668   +1.150984       GPT: RBF+Periodic        0.3987
  3   +2.5494   +1.188480        GPT: Matern-5/2         0.3113
  4   +2.6771   +1.222796            GPT: RBF            0.2514
  5   +2.6245   +1.265410       GPT: RBF+Periodic        0.3378
  6   +2.7222   +1.098804            GPT: RBF            0.2585
  7   +2.4218   +1.053548       GPT: RBF+Periodic        0.2950
  8   +2.7672   +1.028566       GPT: RBF+Periodic        0.2969
  9  